# Ingesta de personas (incremental)

Lee `person.json` (JSON) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.persons` con `MERGE` (llave `person_id` + `file_date`), particionada por `file_date`.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, concat, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType 

### Lectura del archivo

Lectura inicial de `person.json` con inferencia para inspeccionar su estructura anidada.

In [0]:
person_df = (
    spark.read
        .format("json")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/person.json")
)

### Exploración del campo anidado

Prueba la extracción de `forename` y `surname` desde `personName` y su concatenación en `fullname`.

In [0]:
(
    person_df.select(
        col("personId").alias("person_id"),
        col("personName.forename").alias("forename"),
        col("personName.surname").alias("surname")
    )
    .withColumn("fullname",
        concat(col("forename"), lit(" "), col("surname"))
    )
    .show()
)

### Lectura con esquema explícito

Define `personName` como `StructType` anidado y relee el archivo.

In [0]:
personName_shema = StructType([
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])
person_schema = StructType([
    StructField("personId", IntegerType(), True),
    StructField("personName", personName_shema, True)
])
person_df = (
    spark.read
        .schema(person_schema)
        .format("json")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/person.json")
)

### Transformación de columnas

Une `forename` y `surname` en `fullname`, renombra `personId` y agrega `ingestion_date`, `env` y `file_date`.

In [0]:
person_df = (
    add_ingestion_date(person_df)
        .select(
            col("personId").alias("person_id"),
            concat(col("personName.forename"), lit(" "), col("personName.surname")).alias("fullname"),
            lit(v_env).alias("env"),
            lit(v_file_date).alias("file_date")
    ).drop("personName")
)

### Escritura en Silver

Las celdas comentadas conservan estrategias anteriores (borrar partición + `append`). La carga activa es `merge_delta_lake` sobre `movie_silver.persons` con la llave `person_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "persons", "file_date", v_file_date)

In [0]:
"""
(
    person_df.write
        .mode("append")
        .partitionBy("file_date")
        .saveAsTable(f"movie_silver.persons")
)
"""


In [0]:
merge_condition = 'trgt.person_id = src.person_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = person_df, schema_name = "movie_silver", table_name = "persons", merge_condition = merge_condition, partition_column = "file_date")

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")